# Desafio prático: Análise Financeira com Python

**Fintech:** ClearBank  

Este notebook lê e valida transações bancárias, calcula métricas financeiras mensais, identifica movimentações suspeitas, exibe um relatório e exporta os resultados em JSON. A solução principal usa apenas módulos nativos do Python para leitura e processamento; `matplotlib` aparece somente no requisito opcional de visualização.

## 1. Arquivos e critérios de validação

Antes de executar, mantenha `transacoes.csv` no mesmo diretório do notebook. O arquivo contém registros válidos e inválidos para testar todos os critérios do desafio. Uma linha é descartada quando possui `id` inválido ou duplicado, `cliente_id` vazio, data fora do formato `AAAA-MM-DD`, tipo diferente de `credito`/`debito` ou valor não numérico/menor ou igual a zero.

In [1]:
import csv
import json
from datetime import datetime
from pathlib import Path

ARQUIVO_CSV = Path("transacoes.csv")
ARQUIVO_JSON = Path("relatorio.json")
ARQUIVO_GRAFICO = Path("grafico.png")
LIMITE_SUSPEITO = 10000.00
COLUNAS_OBRIGATORIAS = {"id", "data", "cliente_id", "tipo", "valor", "descricao", "categoria"}

print("Configuração carregada. Limite suspeito:", LIMITE_SUSPEITO)

Configuração carregada. Limite suspeito: 10000.0


## 2. Validação e limpeza

A função abaixo valida uma única linha. Conversões de `id`, `valor` e `data` são protegidas com `try/except`, e registros inválidos retornam `None` sem interromper o programa.

In [2]:
def validar_transacao(linha, ids_vistos=None):
    ids_vistos = ids_vistos if ids_vistos is not None else set()

    try:
        id_transacao = int(linha.get("id", "").strip())
        if id_transacao <= 0 or id_transacao in ids_vistos:
            return None
    except (ValueError, AttributeError):
        return None

    cliente_id = linha.get("cliente_id", "").strip()
    if not cliente_id:
        return None

    data_texto = linha.get("data", "").strip()
    try:
        data = datetime.strptime(data_texto, "%Y-%m-%d")
        if data.strftime("%Y-%m-%d") != data_texto:
            return None
    except (ValueError, TypeError):
        return None

    tipo = linha.get("tipo", "").strip().lower()
    if tipo not in {"credito", "debito"}:
        return None

    try:
        valor = float(linha.get("valor", "").strip())
        if valor <= 0:
            return None
    except (ValueError, TypeError, AttributeError):
        return None

    ids_vistos.add(id_transacao)
    return {
        "id": id_transacao,
        "data": data,
        "cliente_id": cliente_id,
        "tipo": tipo,
        "valor": valor,
        "descricao": linha.get("descricao", "").strip(),
        "categoria": linha.get("categoria", "").strip(),
    }

print("Função validar_transacao definida.")

Função validar_transacao definida.


In [3]:
# Testes rápidos da validação
linha_valida = {"id": "99", "data": "2026-05-01", "cliente_id": "CLI099", "tipo": "credito", "valor": "100.50", "descricao": "Teste", "categoria": "teste"}
linha_invalida = {**linha_valida, "id": "100", "valor": "abc"}
assert validar_transacao(linha_valida) is not None
assert validar_transacao(linha_invalida) is None
print("Testes de validação aprovados.")

Testes de validação aprovados.


## 3. Leitura do CSV com módulo nativo

`ler_transacoes()` usa `csv.DictReader`, verifica o cabeçalho e trata `FileNotFoundError`. Cada linha é validada de forma independente.

In [4]:
def ler_transacoes(caminho=ARQUIVO_CSV):
    transacoes_validas = []
    ids_vistos = set()
    total_linhas = 0
    total_invalidas = 0

    try:
        with open(caminho, mode="r", encoding="utf-8-sig", newline="") as arquivo:
            leitor = csv.DictReader(arquivo)
            colunas = set(leitor.fieldnames or [])
            faltantes = COLUNAS_OBRIGATORIAS - colunas
            if faltantes:
                print("CSV inválido. Colunas ausentes:", ", ".join(sorted(faltantes)))
                return [], 0, 0

            for linha in leitor:
                total_linhas += 1
                transacao = validar_transacao(linha, ids_vistos)
                if transacao is None:
                    total_invalidas += 1
                    continue
                transacoes_validas.append(transacao)
    except FileNotFoundError:
        print(f"Arquivo não encontrado: {caminho}")
        return [], 0, 0

    print("===== RESUMO DA LIMPEZA =====")
    print(f"Total de linhas lidas: {total_linhas}")
    print(f"Linhas válidas: {len(transacoes_validas)}")
    print(f"Linhas inválidas: {total_invalidas}")
    return transacoes_validas, total_linhas, total_invalidas

print("Função ler_transacoes definida.")

Função ler_transacoes definida.


In [5]:
# Teste rápido da leitura
transacoes_teste, total_teste, invalidas_teste = ler_transacoes()
assert total_teste == 23
assert len(transacoes_teste) == 16
assert invalidas_teste == 7
print("Teste de leitura aprovado.")

===== RESUMO DA LIMPEZA =====
Total de linhas lidas: 23
Linhas válidas: 16
Linhas inválidas: 7
Teste de leitura aprovado.


## 4. Agrupamento mensal, datas e métricas

`gerar_relatorio()` agrupa por `AAAA-MM`, calcula todas as métricas obrigatórias, mede o período analisado e separa as transações acima de `LIMITE_SUSPEITO`.

In [6]:
def gerar_relatorio(transacoes, total_invalidas):
    if not transacoes:
        return {
            "gerado_em": datetime.now().strftime("%Y-%m-%d"),
            "total_transacoes_validas": 0,
            "total_transacoes_invalidas": total_invalidas,
            "periodo_analisado": None,
            "resumo_mensal": {},
            "transacoes_suspeitas": [],
        }

    resumo_mensal = {}
    suspeitas = []

    for transacao in transacoes:
        mes = transacao["data"].strftime("%Y-%m")
        if mes not in resumo_mensal:
            resumo_mensal[mes] = {
                "quantidade": 0,
                "total_credito": 0.0,
                "total_debito": 0.0,
                "_transacoes": [],
            }

        dados_mes = resumo_mensal[mes]
        dados_mes["quantidade"] += 1
        dados_mes[f"total_{transacao['tipo']}"] += transacao["valor"]
        dados_mes["_transacoes"].append(transacao)

        if transacao["valor"] > LIMITE_SUSPEITO:
            suspeitas.append({
                "id": transacao["id"],
                "cliente_id": transacao["cliente_id"],
                "data": transacao["data"].strftime("%Y-%m-%d"),
                "valor": transacao["valor"],
            })

    for dados_mes in resumo_mensal.values():
        itens = dados_mes.pop("_transacoes")
        maior = max(itens, key=lambda item: item["valor"])
        menor = min(itens, key=lambda item: item["valor"])
        soma_valores = sum(item["valor"] for item in itens)
        dados_mes["total_credito"] = round(dados_mes["total_credito"], 2)
        dados_mes["total_debito"] = round(dados_mes["total_debito"], 2)
        dados_mes["saldo"] = round(dados_mes["total_credito"] - dados_mes["total_debito"], 2)
        dados_mes["media_transacao"] = round(soma_valores / dados_mes["quantidade"], 2)
        dados_mes["maior_transacao"] = {"id": maior["id"], "valor": maior["valor"]}
        dados_mes["menor_transacao"] = {"id": menor["id"], "valor": menor["valor"]}

    data_inicial = min(item["data"] for item in transacoes)
    data_final = max(item["data"] for item in transacoes)

    return {
        "gerado_em": datetime.now().strftime("%Y-%m-%d"),
        "total_transacoes_validas": len(transacoes),
        "total_transacoes_invalidas": total_invalidas,
        "periodo_analisado": {
            "inicio": data_inicial.strftime("%Y-%m-%d"),
            "fim": data_final.strftime("%Y-%m-%d"),
            "dias": (data_final - data_inicial).days,
        },
        "resumo_mensal": dict(sorted(resumo_mensal.items())),
        "transacoes_suspeitas": suspeitas,
    }

print("Função gerar_relatorio definida.")

Função gerar_relatorio definida.


In [7]:
# Teste rápido das métricas
relatorio_teste = gerar_relatorio(transacoes_teste, invalidas_teste)
assert len(relatorio_teste["resumo_mensal"]) == 4
assert len(relatorio_teste["transacoes_suspeitas"]) == 2
assert relatorio_teste["resumo_mensal"]["2026-01"]["quantidade"] == 4
print("Testes de métricas aprovados.")

Testes de métricas aprovados.


## 5. Exportação JSON e exibição no terminal

As funções seguintes mantêm separadas as responsabilidades de salvar e apresentar o relatório.

In [8]:
def salvar_json(relatorio, caminho=ARQUIVO_JSON):
    with open(caminho, mode="w", encoding="utf-8") as arquivo:
        json.dump(relatorio, arquivo, ensure_ascii=False, indent=2)


def formatar_moeda(valor):
    formatado = f"{valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
    return f"R$ {formatado}"


def exibir_relatorio(relatorio):
    periodo = relatorio["periodo_analisado"]
    print("\n===== RELATÓRIO FINANCEIRO CLEARBANK =====")
    if periodo:
        print(f"Período analisado: {periodo['inicio']} → {periodo['fim']} ({periodo['dias']} dias)")
    print(f"Transações válidas: {relatorio['total_transacoes_validas']}")
    print(f"Transações inválidas: {relatorio['total_transacoes_invalidas']}")

    for mes, dados in relatorio["resumo_mensal"].items():
        print(f"\n===== MÊS: {mes} =====")
        print(f"Transações: {dados['quantidade']}")
        print(f"Total crédito: {formatar_moeda(dados['total_credito'])}")
        print(f"Total débito: {formatar_moeda(dados['total_debito'])}")
        print(f"Saldo: {formatar_moeda(dados['saldo'])}")
        print(f"Média: {formatar_moeda(dados['media_transacao'])}")
        print(f"Maior valor: {formatar_moeda(dados['maior_transacao']['valor'])}")
        print(f"Menor valor: {formatar_moeda(dados['menor_transacao']['valor'])}")

    print("\n===== TRANSAÇÕES SUSPEITAS =====")
    suspeitas = relatorio["transacoes_suspeitas"]
    if not suspeitas:
        print("Nenhuma transação suspeita encontrada.")
    for item in suspeitas:
        print(
            f"ID: {item['id']} | Cliente: {item['cliente_id']} | "
            f"Data: {item['data']} | Valor: {formatar_moeda(item['valor'])}"
        )

print("Funções de saída definidas.")

Funções de saída definidas.


## 6. Requisito opcional: gráfico com matplotlib

O gráfico de barras mostra o saldo mensal e é salvo exatamente como `grafico.png`.

In [9]:
def gerar_grafico(relatorio, caminho=ARQUIVO_GRAFICO):
    import matplotlib.pyplot as plt

    meses = list(relatorio["resumo_mensal"].keys())
    saldos = [relatorio["resumo_mensal"][mes]["saldo"] for mes in meses]
    cores = ["#22c55e" if saldo >= 0 else "#ef4444" for saldo in saldos]

    figura, eixo = plt.subplots(figsize=(9, 5))
    eixo.bar(meses, saldos, color=cores, label="Saldo mensal")
    eixo.axhline(0, color="#334155", linewidth=0.8)
    eixo.set_title("ClearBank — saldo mensal")
    eixo.set_xlabel("Mês")
    eixo.set_ylabel("Saldo (R$)")
    eixo.legend()
    eixo.grid(axis="y", alpha=0.2)
    figura.tight_layout()
    figura.savefig(caminho, dpi=150)
    plt.close(figura)
    return caminho

print("Função gerar_grafico definida.")

Função gerar_grafico definida.


## 7. Execução principal

A função principal conecta as etapas somente depois que cada parte foi implementada e testada.

In [10]:
def main():
    transacoes, _, total_invalidas = ler_transacoes()
    if not transacoes:
        print("Nenhuma transação válida disponível para análise.")
        return None

    relatorio = gerar_relatorio(transacoes, total_invalidas)
    salvar_json(relatorio)
    gerar_grafico(relatorio)
    exibir_relatorio(relatorio)
    print(f"\nRelatório JSON salvo em: {ARQUIVO_JSON}")
    print(f"Gráfico salvo em: {ARQUIVO_GRAFICO}")
    return relatorio

print("Função main definida.")

Função main definida.


In [11]:
relatorio = main()
assert relatorio is not None
assert ARQUIVO_JSON.exists()
assert ARQUIVO_GRAFICO.exists()
print("Execução principal concluída sem erros.")

===== RESUMO DA LIMPEZA =====
Total de linhas lidas: 23
Linhas válidas: 16
Linhas inválidas: 7

===== RELATÓRIO FINANCEIRO CLEARBANK =====
Período analisado: 2026-01-05 → 2026-04-25 (110 dias)
Transações válidas: 16
Transações inválidas: 7

===== MÊS: 2026-01 =====
Transações: 4
Total crédito: R$ 17.000,00
Total débito: R$ 410,35
Saldo: R$ 16.589,65
Média: R$ 4.352,59
Maior valor: R$ 12.500,00
Menor valor: R$ 89,90

===== MÊS: 2026-02 =====
Transações: 4
Total crédito: R$ 4.500,00
Total débito: R$ 1.140,29
Saldo: R$ 3.359,71
Média: R$ 1.410,07
Maior valor: R$ 4.500,00
Menor valor: R$ 149,99

===== MÊS: 2026-03 =====
Transações: 4
Total crédito: R$ 22.000,00
Total débito: R$ 1.610,25
Saldo: R$ 20.389,75
Média: R$ 5.902,56
Maior valor: R$ 17.500,00
Menor valor: R$ 410,25

===== MÊS: 2026-04 =====
Transações: 4
Total crédito: R$ 4.600,00
Total débito: R$ 980,00
Saldo: R$ 3.620,00
Média: R$ 1.395,00
Maior valor: R$ 4.600,00
Menor valor: R$ 99,90

===== TRANSAÇÕES SUSPEITAS =====
ID: 4 | Cl